In [1]:
import openai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import re
import time
from openai import OpenAI
import json
import pickle
import os

sys.path += ['../src/']

#from model import *
import model as mod
import model
from utils import *
from plotting_functions import *
from matplotlib.gridspec import GridSpec
from itertools import product
import statsmodels.api as sm

In [73]:
file_name="../data/HeemeijeretalJEDC2009-Feedback Experiment - Prices, Predictions.xls"
a=pd.read_excel(file_name)

In [84]:
# Prepare an empty DataFrame for long format data
long_format_data = []

# Number of time points
n_time_points = 50

# Iterate over the number of experiments
for n_experiment in range(13):
    p_exp = a.iloc[4:4+n_time_points, 7*n_experiment].astype(float).values
    pe_exp = a.iloc[4:4+n_time_points, np.array(range(1, 7)) + 7*n_experiment].astype(float).values
    
    # Determine treatment type
    if n_experiment < 6:
        treatment = "negative"
    else:
        treatment = "positive"
    
    # Collect data for the current experiment
    for t in range(n_time_points):
        for j in range(6):  # There are 6 participants
            long_format_data.append([n_experiment + 1, treatment, t + 1, p_exp[t], j + 1, pe_exp[t][j]])

# Create DataFrame from long format data
long_df = pd.DataFrame(long_format_data, columns=['experiment', 'feedback', 'time', 'p', 'participant', 'pexp'])

# Display the long format DataFrame
df = long_df

In [85]:
excluded_periods_negative = {
    1: [44],
    2: [8, 9, 10, 11, 12],
    3: [36, 45, 46, 47, 48, 49, 50],
    4: [21, 41],
    5: [8, 9, 10, 11, 12, 13, 14, 15, 22]
}
excluded_periods_positive = {
    9: [39],
    10: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 28, 34, 45],
    11: [15, 25, 35]
}

# Set values to NaN for negative feedback groups
for exp, periods in excluded_periods_negative.items():
    for period in periods:
        df.loc[(df['experiment'] == exp) & (df['time'] == period), ['p', 'pexp']] = np.nan

# Set values to NaN for positive feedback groups
for exp, periods in excluded_periods_positive.items():
    for period in periods:
        df.loc[(df['experiment'] == exp + 6) & (df['time'] == period), ['p', 'pexp']] = np.nan


learning_phase_ends = []

for n_experiment in range(1, 14):
    experiment_data = df[df['experiment'] == n_experiment]
    for t in range(1, n_time_points + 1):
        time_data = experiment_data[experiment_data['time'] == t]
        if time_data['p'].notna().all() and time_data['pexp'].notna().all():
            p = time_data['p'].iloc[0]
            within_5_percent = (np.abs(time_data['pexp'] - p) / p) <= 0.05
            if within_5_percent.sum() >= 4:
                learning_phase_ends.append((n_experiment, t))
                break

learning_phase_df = pd.DataFrame(learning_phase_ends, columns=['experiment', 'learning_phase_end'])

for _, row in learning_phase_df.iterrows():
    exp = row['experiment']
    end_time = row['learning_phase_end']
    df.loc[(df['experiment'] == exp) & (df['time'] <= end_time), ['p', 'pexp']] = np.nan


In [86]:
df

,experiment,feedback,time,p,participant,pexp
0,1,negative,1,NaN,1,NaN
1,1,negative,1,NaN,2,NaN
2,1,negative,1,NaN,3,NaN
3,1,negative,1,NaN,4,NaN
4,1,negative,1,NaN,5,NaN
...,...,...,...,...,...,...
3895,13,positive,50,63.03,2,62.15
3896,13,positive,50,63.03,3,62.55
3897,13,positive,50,63.03,4,62.50
3898,13,positive,50,63.03,5,62.75


In [87]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox

# Assuming df is already created and values are set to NaN for learning phases and outliers

# Function to prepare data and fit the model for each experiment and participant
def fit_time_series_model_iterative(participant_data, experiment, participant):

    participant_data = participant_data.copy()
    
    # Create lagged variables
    for lag in range(1, 4):
        participant_data[f'p{lag}'] = participant_data['p'].shift(lag)
        participant_data[f'p{lag}e'] = participant_data['pexp'].shift(lag)
    
    # Drop rows with NaN values resulting from the lagging process
    participant_data.dropna(inplace=True)
    
    # Define the independent variables (lags) and dependent variable
    X = participant_data[[f'p{i}' for i in range(1, 4)] + [f'p{i}e' for i in range(1, 4)]]
    y = participant_data['pexp']
    
    # Add a constant term to the independent variables
    X = sm.add_constant(X)
    
    # Fit the initial linear regression model
    model = sm.OLS(y, X).fit()
    
    # Iterative procedure to eliminate insignificant variables
    while True:
        # Check if all p-values are below 5%
        if (model.pvalues < 0.05).all():
            break
        
        # Get the variable with the largest p-value
        max_p_var = model.pvalues.idxmax()
        
        # Drop the variable with the largest p-value from X
        X.drop(columns=[max_p_var], inplace=True)
        
        # Fit the updated model
        model = sm.OLS(y, X).fit()
    
    # Initialize a dictionary to store parameters
    params_dict = {
        'Experiment': experiment,
        'Participant': participant,
        'const': 0,
        'p1': 0,
        'p2': 0,
        'p3': 0,
        'p1e': 0,
        'p2e': 0,
        'p3e': 0,
        'R2': model.rsquared,
        'AC': "Yes", #if any(acorr_ljungbox(model.resid, lags=20)[1] < 0.05) else "No",
        'Eq.': 1,#model.params[-1] if len(model.params) > 0 else 0,
        'MSE': np.mean(model.resid ** 2)
    }
    
    # Update dictionary with significant parameters
    for param in model.params.index:
        params_dict[param] = model.params[param]
    
    return params_dict

# Initialize a list to store results
results = []

# Iterate over each experiment and participant
for experiment in df['experiment'].unique():
    for participant in df[df['experiment'] == experiment]['participant'].unique():
        # Filter the data for the specific experiment and participant
        participant_data = df[(df['experiment'] == experiment) & (df['participant'] == participant)].copy()

        # Fit the iterative model and get parameters
        params_dict = fit_time_series_model_iterative(participant_data, experiment, participant)
        
        # Append results to the list
        results.append(params_dict)

# Create a DataFrame from the results list
results_df = pd.DataFrame(results)

# Set Participant column as index (if desired)
#results_df.set_index('Participant', inplace=True)

# Display the results DataFrame
#print(results_df)
# Create a DataFrame from the results list
#results_df = pd.DataFrame(results, columns=['Participant', 'c', 'p1', 'p2', 'p3', 'p1e', 'p2e', 'p3e', 'R2', 'AC', 'Eq.', 'MSE'])

# Display the results DataFrame
#print(results_df)


C:\Users\marco\anaconda3\envs\py310\lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
C:\Users\marco\anaconda3\envs\py310\lib\site-packages\statsmodels\regression\linear_model.py:1781: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


In [88]:
results_df

,Experiment,Participant,const,p1,p2,p3,p1e,p2e,p3e,R2,AC,Eq.,MSE
0,1,1,52.860478,0.122782,0.000000,0.0,0.000000,0.000000,0.000000,2.146357e-01,Yes,1,0.018745
1,1,2,24.552291,0.679913,-0.347975,0.0,0.402703,-0.143713,0.000000,8.764646e-01,Yes,1,0.023294
2,1,3,60.059211,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,-2.220446e-16,Yes,1,0.197707
3,1,4,0.000000,0.778265,0.221717,0.0,0.000000,0.000000,0.000000,9.999860e-01,Yes,1,0.050456
4,1,5,0.000000,0.612052,0.000000,0.0,0.389959,0.000000,0.000000,9.999244e-01,Yes,1,0.274147
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,13,2,0.000000,0.997959,0.000000,0.0,0.000000,0.000000,0.000000,9.999892e-01,Yes,1,0.041288
74,13,3,6.500721,1.131479,-0.235863,0.0,0.000000,0.000000,0.000000,9.655858e-01,Yes,1,0.019532
75,13,4,0.000000,1.067203,0.000000,0.0,0.000000,-0.162859,0.096463,9.999955e-01,Yes,1,0.017210
76,13,5,0.000000,1.398625,-0.397940,0.0,0.000000,0.000000,0.000000,9.999889e-01,Yes,1,0.042856


In [98]:
agent_id

6

In [100]:
temperature = 0.3
memory = 1

results = []

for feedback in ["neg","pos"]:
    for expmnt_num in range(1,4):
        for agent_id in range(0,6):
    
            df = pd.read_csv("../results/experiments/expmnt_results_"+feedback+"_original_30-50w_num_"+str(expmnt_num)+\
                             "_temp_"+str(temperature).replace('.', '-')+"_memory_"+str(memory)+"_ntime_50_nagents_6.csv")
            
            df = df.loc[df["agent_id"]==agent_id,["time_step","actual_price","agent_id","predicted_price"]]
            df = df.rename(columns={"time_step": "time", "actual_price": "p", "agent_id": "participant", "predicted_price": "pexp"})
            
            df["experiment"] = 1
            df["feedback"] = feedback
    
            params_dict = fit_time_series_model_iterative(df, expmnt_num, agent_id)
    
            results.append(params_dict)

In [101]:
pd.DataFrame(results)

,Experiment,Participant,const,p1,p2,p3,p1e,p2e,p3e,R2,AC,Eq.,MSE
0,1,0,89.909230,-0.507297,-0.322220,0.334363,0.000000,0.000000,0.000000,0.853839,Yes,1,0.269027
1,1,1,42.616602,0.375053,0.000000,-0.240443,0.295032,-0.133453,0.000000,0.468170,Yes,1,0.285910
2,1,2,16.612066,0.996995,-0.599878,0.000000,0.329502,0.000000,0.000000,0.657337,Yes,1,0.196589
3,1,3,0.000000,0.325198,0.270447,0.240247,0.000000,0.174622,0.000000,0.999944,Yes,1,0.203656
4,1,4,0.000000,0.551105,0.000000,0.000000,0.000000,0.241555,0.212079,0.999924,Yes,1,0.272524
5,1,5,12.156539,0.000000,0.000000,0.455690,0.000000,0.347005,0.000000,0.864658,Yes,1,0.250614
6,2,0,36.565373,0.616217,0.220506,-0.204461,0.000000,0.000000,-0.230210,0.812396,Yes,1,0.421448
7,2,1,-26.910178,0.818172,0.165033,0.171628,0.000000,0.000000,0.304272,0.815053,Yes,1,0.601256
8,2,2,183.840794,-0.375730,-1.433358,0.000000,0.000000,-0.544835,0.284226,0.818448,Yes,1,1.005350
9,2,3,150.121547,-0.468123,-1.316677,0.358153,0.000000,-0.605760,0.531212,0.710656,Yes,1,1.686874
